In [0]:
data = [(1, "India", 100),
        (2, "US", 200),
        (3, "India", 300),
        (4, "China", 400),
        (5, "US", 500),
        (6, "China", 600),
        (7, "India", 700)]

columns = ["id", "country", "amount"]

df = spark.createDataFrame(data, columns)

df.write.format("delta").save("/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice")

In [0]:
spark.read.format("delta").load("/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice").show()

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice" + "/_delta_log"))

In [0]:
df2 = spark.createDataFrame([(8, "UK", 800)], columns)
df2.write.format("delta").mode("append").save("/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice")

In [0]:
spark.read.format("delta").load("/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice").show()

In [0]:
from delta.tables import DeltaTable
deltatable = DeltaTable.forPath(spark, "/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice")
deltatable.history().show()

In [0]:
spark.read.format("delta").option("versionAsOf", 0).load("/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice").show()

In [0]:
df3 = spark.createDataFrame([(9, "Japan", 900, "Yen")], ["id", "country", "amount", "currency"])
df3.write.format("delta").option("mergeSchema", "true").mode("append").save("/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice")

In [0]:
spark.read.format("delta").load("/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice").show()

In [0]:
from delta.tables import DeltaTable
deltatable = DeltaTable.forPath(spark, "/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice")
deltatable.update(condition = "id = 1", set = {"amount" : "200"})

In [0]:
deltatable.delete("id = 7")

In [0]:
spark.read.format("delta").load("/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice").show()

In [0]:
df4 = [(1, "India", 300) ,(10, "China", 1000)]
df4_new = spark.createDataFrame(df4, columns)

In [0]:
deltatable.alias("target").merge(df4_new.alias("source"), "target.id = source.id") \
  .whenMatchedUpdate(set = {"country": "source.country", "amount": "source.amount"}) \
  .whenNotMatchedInsert(values = {"id": "source.id", "country": "source.country", "amount": "source.amount"}) \
  .execute()

In [0]:
spark.sql("OPTIMIZE delta.`/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice`")

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice`

In [0]:
from delta.tables import DeltaTable
deltatable = DeltaTable.forPath(spark, "/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice")
deltatable.history().show()

In [0]:
%sql
optimize delta.`/Volumes/workspace/delta_lake/delta_lake_lab/delta_practice`
ZORDER BY (country)